# E-Commerce Sales & Customer Analysis
### Intermediate Python Data Analyst Portfolio Project

**Objective:** Analyze transaction-level e-commerce data to identify sales trends, profitability drivers, customer value, and regional/category performance.

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn

**Analyst workflow:** Data quality → KPI development → EDA → customer/product/region analysis → business insights → recommendations.


## 1. Business Questions

This analysis is designed to answer:

1. What are the overall revenue, profit, order, and customer KPIs?
2. How are revenue and profit changing over time?
3. Which product categories drive revenue and profit?
4. Which regions contribute the most profit?
5. Which customers have the highest lifetime revenue in the dataset?
6. How do discounts relate to profit margin?
7. What actions could management consider based on the evidence?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

df = pd.read_csv("../data/ecommerce_sales.csv")
df.head()


## 2. Data Quality Check

Before analysis, inspect data types, missing values, duplicate records, and basic descriptive statistics. Cleaning is important because inaccurate records can change KPIs and business conclusions.


In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False))

print("\nDuplicate rows:", df.duplicated().sum())

df.describe(include="all").T


In [ ]:
# Clean data
df["order_date"] = pd.to_datetime(df["order_date"])

before = len(df)
df = df.drop_duplicates().copy()
duplicates_removed = before - len(df)

df["region"] = df["region"].fillna(df["region"].mode()[0])
df["discount"] = df["discount"].fillna(df["discount"].median())

# Orders with no customer_id can't be tied to a real customer. We keep them
# in revenue/profit totals (the sale still happened) but exclude them from
# customer-level rankings below — otherwise every anonymous order gets lumped
# into one fake "UNKNOWN" customer who then looks like a top account.
df["customer_id"] = df["customer_id"].fillna("UNKNOWN")
known_customers = df[df["customer_id"] != "UNKNOWN"].copy()
missing_customer_orders = int((df["customer_id"] == "UNKNOWN").sum())

# Feature engineering
df["month"] = df["order_date"].dt.to_period("M").astype(str)
df["profit_margin"] = np.where(df["revenue"] != 0, df["profit"] / df["revenue"], 0)

print(f"Duplicate rows removed: {duplicates_removed}")
print(f"Orders with missing customer_id (kept in totals, excluded from customer ranking): {missing_customer_orders}")
print("Remaining missing values:", int(df.isna().sum().sum()))

## 3. KPI Overview

The main KPIs provide a high-level view of the business before drilling into individual dimensions.


In [ ]:
kpis = pd.Series({
    "Revenue": df["revenue"].sum(),
    "Profit": df["profit"].sum(),
    "Profit Margin": df["profit"].sum() / df["revenue"].sum(),
    "Orders": df["order_id"].nunique(),
    "Known Customers": known_customers["customer_id"].nunique(),
    "Units Sold": df["quantity"].sum(),
    "Average Order Value": df["revenue"].sum() / df["order_id"].nunique()
})

kpis

## 4. Monthly Sales & Profit Trend

Time-series analysis helps identify growth patterns, seasonal behavior, and periods that deserve further investigation.


In [ ]:
monthly = (
    df.groupby("month", as_index=False)
      .agg(revenue=("revenue", "sum"),
           profit=("profit", "sum"),
           orders=("order_id", "nunique"))
)

monthly.head()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly["month"], monthly["revenue"], marker="o", label="Revenue")
ax.plot(monthly["month"], monthly["profit"], marker="o", label="Profit")
ax.set_title("Monthly Revenue and Profit")
ax.set_xlabel("Month")
ax.set_ylabel("USD")
ax.tick_params(axis="x", rotation=60)
ax.legend()
plt.tight_layout()
plt.show()


## 5. Category Performance

Compare categories using revenue, profit, units sold, and profit margin. High revenue does not necessarily mean high profitability.


In [ ]:
category = (
    df.groupby("category", as_index=False)
      .agg(revenue=("revenue", "sum"),
           profit=("profit", "sum"),
           units=("quantity", "sum"))
)
category["profit_margin"] = category["profit"] / category["revenue"]
category.sort_values("revenue", ascending=False)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=category.sort_values("revenue", ascending=False),
            x="revenue", y="category", ax=ax)
ax.set_title("Revenue by Product Category")
ax.set_xlabel("Revenue (USD)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=category.sort_values("profit_margin", ascending=False),
            x="profit_margin", y="category", ax=ax)
ax.set_title("Profit Margin by Product Category")
ax.set_xlabel("Profit Margin")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 6. Regional Performance

Regional analysis can reveal where revenue is being generated and whether strong sales are translating into profit.


In [ ]:
regional = (
    df.groupby("region", as_index=False)
      .agg(revenue=("revenue", "sum"),
           profit=("profit", "sum"),
           orders=("order_id", "nunique"))
)
regional["profit_margin"] = regional["profit"] / regional["revenue"]
regional.sort_values("profit", ascending=False)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=regional.sort_values("profit", ascending=False),
            x="profit", y="region", ax=ax)
ax.set_title("Profit by Region")
ax.set_xlabel("Profit (USD)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 7. Customer Value Analysis

Customer-level aggregation helps identify high-value customers and supports customer retention or segmentation decisions.


In [ ]:
customer = (
    known_customers.groupby("customer_id", as_index=False)
      .agg(revenue=("revenue", "sum"),
           profit=("profit", "sum"),
           orders=("order_id", "nunique"),
           units=("quantity", "sum"))
)

customer["avg_order_value"] = customer["revenue"] / customer["orders"]

top_customers = customer.sort_values("revenue", ascending=False).head(10)
top_customers

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=top_customers, x="revenue", y="customer_id", ax=ax)
ax.set_title("Top 10 Customers by Revenue")
ax.set_xlabel("Revenue (USD)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## 8. Discount & Profitability Analysis

Discounting can increase sales volume but reduce profitability. This section checks whether higher discounts are associated with lower profit margins in the observed transactions.


In [ ]:
discount_bins = pd.cut(
    df["discount"],
    bins=[-0.001, 0.05, 0.10, 0.15, 0.20, 0.35],
    labels=["0–5%", "5–10%", "10–15%", "15–20%", "20%+"]
)

discount_analysis = (
    df.assign(discount_band=discount_bins)
      .groupby("discount_band", observed=False)
      .agg(revenue=("revenue", "sum"),
           profit=("profit", "sum"),
           orders=("order_id", "nunique"))
      .reset_index()
)
discount_analysis["profit_margin"] = discount_analysis["profit"] / discount_analysis["revenue"]
discount_analysis


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=discount_analysis, x="discount_band",
            y="profit_margin", ax=ax)
ax.set_title("Profit Margin by Discount Band")
ax.set_xlabel("Discount")
ax.set_ylabel("Profit Margin")
plt.tight_layout()
plt.show()


## 9. Product-Level Performance

A product can have strong revenue but weak margins. Reviewing product-level metrics helps identify products that may need pricing, cost, or promotion review.


In [ ]:
product = (
    df.groupby(["category", "product"], as_index=False)
      .agg(revenue=("revenue", "sum"),
           profit=("profit", "sum"),
           units=("quantity", "sum"))
)
product["profit_margin"] = product["profit"] / product["revenue"]

product.sort_values("profit", ascending=False).head(10)


## 10. Key Findings

Use the outputs above to write concise, evidence-based findings. A strong portfolio project should distinguish **what the data shows** from **what management could consider doing**.

Example structure:

- **Sales trend:** Identify the strongest and weakest periods from the monthly table/chart.
- **Category:** Compare revenue and margin rather than relying on revenue alone.
- **Region:** Identify regions with meaningful profit contribution and compare their margins.
- **Customers:** Use the top-customer table to discuss concentration of revenue.
- **Discounts:** Compare discount bands and profit margins before making pricing recommendations.


In [ ]:
# Automatically generate a compact findings table for reference
summary = {
    "Highest revenue category": category.loc[category["revenue"].idxmax(), "category"],
    "Highest profit category": category.loc[category["profit"].idxmax(), "category"],
    "Highest margin category": category.loc[category["profit_margin"].idxmax(), "category"],
    "Highest profit region": regional.loc[regional["profit"].idxmax(), "region"],
    "Top customer by revenue": top_customers.iloc[0]["customer_id"],
    "Top customer revenue": top_customers.iloc[0]["revenue"],
}
pd.Series(summary)


## 11. Business Recommendations

Based on the measured results, management could consider:

1. **Prioritize profitable categories:** Allocate marketing and inventory attention using both revenue and profit margin.
2. **Review discount strategy:** Investigate discount bands associated with weaker margins before expanding promotions.
3. **Retain high-value customers:** Develop targeted retention strategies for customers contributing substantial revenue.
4. **Investigate regional differences:** Compare regional product mix, pricing, and fulfillment costs to understand profit variation.
5. **Monitor monthly KPIs:** Track revenue, profit, orders, and margin regularly to detect changes early.

These are analytical recommendations; they should be validated with additional information such as marketing spend, acquisition cost, inventory levels, and customer acquisition data.


## 12. Conclusion

This project demonstrates an end-to-end Python data analytics workflow. The analysis combines data cleaning, feature engineering, KPI development, exploratory data analysis, visualization, customer analysis, and business interpretation.

The key portfolio takeaway is not only the Python code, but the ability to translate transaction data into measurable business questions and actionable analytical insights.
